# Black-Box (Hard-Label) Adversarial Attack on a Pretrained CNN

**Method:** HopSkipJump-style decision-based attack (untargeted, L2), implemented from scratch.
Only the predicted **label** of the CNN is ever used: no scores, gradients, weights or architecture.

Pipeline per image: random-noise initialisation -> binary search to the decision boundary -> repeat
(gradient-direction estimation from label queries, geometric step, binary search) -> 8-bit quantisation and
re-verification on the exact PNG that is saved.

> **Before running:** edit the *Configuration* cell (paths) and, if the shared notebook applies any
> normalisation / resizing before the model, copy it into `preprocess()`.

## 1. Configuration (final parameters)

In [ ]:
MODEL_PATH   = "model.pt"              # pretrained CNN shared by the organisers
CLEAN_DIR    = "clean_images"          # unzipped folder with the 1000 clean .png files
ADV_DIR      = "adversarial_images"    # output folder (one adversarial PNG per clean PNG)
RESULTS_CSV  = "results.csv"

# --- attack parameters (final) ---
N_ITERS      = 10      # HopSkipJump iterations
INIT_EVALS   = 20      # initial number of label queries for each gradient estimate
MAX_EVALS    = 200     # cap on queries per gradient estimate
GAMMA        = 1.0     # probe-radius scale
BIN_TOL      = 1e-3    # binary-search tolerance (fraction of the segment)
MAX_RETRIES  = 3       # re-run an image with a new seed / more iterations if it fails
SEED         = 0
BATCH        = 64      # batch size used when sending queries to the model

## 2. Load model and define the label-only oracle

In [ ]:
import os, glob, time, csv
import numpy as np
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    model = torch.load(MODEL_PATH, map_location=device, weights_only=False)
except Exception:
    model = torch.jit.load(MODEL_PATH, map_location=device)   # TorchScript fallback
model.eval()

def preprocess(x):
    """x: float tensor (N,C,H,W) in [0,1].
    EDIT to match the shared notebook (e.g. mean/std normalisation or resizing). Identity by default."""
    return x

@torch.no_grad()
def label_fn(x_np):
    """Hard-label oracle: returns ONLY predicted class indices (scores are discarded)."""
    out = []
    for i in range(0, len(x_np), BATCH):
        xb = torch.from_numpy(x_np[i:i + BATCH]).to(device)
        out.append(model(preprocess(xb)).argmax(dim=1).cpu().numpy())
    return np.concatenate(out)

## 3. Attack implementation

In [ ]:
import numpy as np

class QueryCounter:
    """Wraps a label-only oracle and counts every image sent to the model."""
    def __init__(self, label_fn):
        self.label_fn = label_fn      # f(batch float32 (N,C,H,W) in [0,1]) -> int labels (N,)
        self.n = 0
    def __call__(self, x):
        x = np.asarray(x, dtype=np.float32)
        if x.ndim == 3:
            x = x[None]
        self.n += len(x)
        return self.label_fn(x)

def is_adv(oracle, x, y0):
    return oracle(x) != y0                       # (N,) bool

def binary_search_to_boundary(oracle, x0, x_adv, y0, tol=1e-3, max_steps=20):
    """Move x_adv toward x0 along the line, keeping it adversarial. Returns adversarial-side point."""
    lo, hi = 0.0, 1.0                            # blend weight on x_adv: lo=non-adv, hi=adv
    steps = 0
    while hi - lo > tol and steps < max_steps:
        mid = (lo + hi) / 2
        cand = (1 - mid) * x0 + mid * x_adv
        if is_adv(oracle, cand, y0)[0]:
            hi = mid
        else:
            lo = mid
        steps += 1
    return (1 - hi) * x0 + hi * x_adv

def find_initial_adversarial(oracle, x0, y0, rng, max_tries=200):
    """Random uniform noise images until one is misclassified, then walk toward x0."""
    for _ in range(max_tries // 20):
        batch = rng.uniform(0, 1, size=(20,) + x0.shape).astype(np.float32)
        adv = is_adv(oracle, batch, y0)
        if adv.any():
            return batch[np.argmax(adv)]
    # fallback: blends of x0 with strong noise
    for a in np.linspace(0.3, 1.0, 8):
        batch = np.clip((1 - a) * x0[None] + a * rng.uniform(0, 1, size=(20,) + x0.shape), 0, 1).astype(np.float32)
        adv = is_adv(oracle, batch, y0)
        if adv.any():
            return batch[np.argmax(adv)]
    return None

def estimate_gradient(oracle, x_b, y0, n_samples, delta, rng):
    d = x_b.size
    u = rng.standard_normal((n_samples,) + x_b.shape).astype(np.float32)
    u /= np.linalg.norm(u.reshape(n_samples, -1), axis=1).reshape((-1,) + (1,) * x_b.ndim)
    pert = np.clip(x_b[None] + delta * u, 0, 1)
    u = (pert - x_b[None]) / delta               # account for clipping
    phi = np.where(is_adv(oracle, pert, y0), 1.0, -1.0).astype(np.float32)
    if phi.mean() == 1.0:
        g = u.mean(0)
    elif phi.mean() == -1.0:
        g = -u.mean(0)
    else:
        phi = phi - phi.mean()                   # baseline variance reduction
        g = (phi.reshape((-1,) + (1,) * x_b.ndim) * u).mean(0)
    n = np.linalg.norm(g)
    return g / (n + 1e-12)

def hopskipjump(oracle, x0, y0, rng, n_iters=8, init_evals=20, max_evals=200,
                gamma=1.0, tol=1e-3):
    """Untargeted, L2, hard-label HopSkipJump attack. Returns adversarial float image or None."""
    d = x0.size
    x_init = find_initial_adversarial(oracle, x0, y0, rng)
    if x_init is None:
        return None
    x_adv = binary_search_to_boundary(oracle, x0, x_init, y0, tol=tol)
    for t in range(1, n_iters + 1):
        dist = np.linalg.norm(x_adv - x0)
        n_samples = int(min(init_evals * np.sqrt(t + 1), max_evals))
        delta = max(gamma * 3 * dist / np.sqrt(d), 1e-3)   # probe radius shrinks as we approach x0
        g = estimate_gradient(oracle, x_adv, y0, n_samples, delta=delta, rng=rng)
        # geometric progression step size
        eps = dist / np.sqrt(t)
        for _ in range(15):
            cand = np.clip(x_adv + eps * g, 0, 1)
            if is_adv(oracle, cand, y0)[0]:
                break
            eps /= 2
        else:
            cand = x_adv
        x_adv = binary_search_to_boundary(oracle, x0, cand, y0, tol=tol)
    return x_adv

def quantize_and_verify(oracle, x0, x_adv, y0, max_push=30, push=0.02):
    """Round to 8-bit and ensure the saved PNG is still misclassified; nudge past the boundary if needed."""
    delta = x_adv - x0
    for k in range(max_push):
        cand = x0 + delta * (1 + push * k)
        q = np.round(np.clip(cand, 0, 1) * 255).astype(np.uint8)
        if is_adv(oracle, q.astype(np.float32) / 255.0, y0)[0]:
            return q
    return None

## 4. Run the attack on all clean images (resumable)

In [ ]:
os.makedirs(ADV_DIR, exist_ok=True)
files = sorted(glob.glob(os.path.join(CLEAN_DIR, "*.png")))
print(f"{len(files)} clean images found")

def load_png(path):
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.float32).transpose(2, 0, 1) / 255.0

done = {}
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV) as f:
        for r in csv.DictReader(f):
            done[r["file"]] = r
fields = ["file", "clean_label", "adv_label", "success", "l2", "linf_255", "queries", "retries"]
new_file = not os.path.exists(RESULTS_CSV)
fh = open(RESULTS_CSV, "a", newline="")
writer = csv.DictWriter(fh, fieldnames=fields)
if new_file:
    writer.writeheader()

t0 = time.time()
for idx, path in enumerate(files):
    name = os.path.basename(path)
    if name in done and os.path.exists(os.path.join(ADV_DIR, name)):
        continue
    x0 = load_png(path)
    oracle = QueryCounter(label_fn)
    y0 = int(oracle(x0)[0])
    q_img, retries = None, 0
    for retries in range(MAX_RETRIES):
        rng = np.random.default_rng(SEED + 1000 * idx + retries)
        adv = hopskipjump(oracle, x0, y0, rng, n_iters=N_ITERS + 5 * retries,
                          init_evals=INIT_EVALS, max_evals=MAX_EVALS, gamma=GAMMA, tol=BIN_TOL)
        if adv is None:
            continue
        q_img = quantize_and_verify(oracle, x0, adv, y0)
        if q_img is not None:
            break
    if q_img is None:                                   # last resort: keep clean image, flagged as failure
        q_img = np.round(x0 * 255).astype(np.uint8)
    Image.fromarray(q_img.transpose(1, 2, 0)).save(os.path.join(ADV_DIR, name))
    xq = q_img.astype(np.float32) / 255.0
    adv_label = int(label_fn(xq[None])[0])              # verification query (not counted in attack budget)
    writer.writerow(dict(file=name, clean_label=y0, adv_label=adv_label, success=int(adv_label != y0),
                         l2=float(np.linalg.norm(xq - x0)), linf_255=float(np.abs(xq - x0).max() * 255),
                         queries=oracle.n, retries=retries))
    fh.flush()
    if (idx + 1) % 25 == 0:
        print(f"{idx + 1}/{len(files)} done  ({time.time() - t0:.0f}s)")
fh.close()

## 5. Independent verification on the saved PNG files

In [ ]:
try:
    from skimage.metrics import structural_similarity as ssim
except Exception:
    ssim = None

succ, l2s, linfs, psnrs, ssims = [], [], [], [], []
for path in files:
    name = os.path.basename(path)
    x0, xa = load_png(path), load_png(os.path.join(ADV_DIR, name))      # reload exactly what will be submitted
    y_clean, y_adv = int(label_fn(x0[None])[0]), int(label_fn(xa[None])[0])
    succ.append(y_clean != y_adv)
    diff = xa - x0
    l2s.append(np.linalg.norm(diff)); linfs.append(np.abs(diff).max() * 255)
    mse = float((diff ** 2).mean()); psnrs.append(99.0 if mse == 0 else 10 * np.log10(1.0 / mse))
    if ssim is not None:
        ssims.append(ssim(x0.transpose(1, 2, 0), xa.transpose(1, 2, 0), channel_axis=2, data_range=1.0))

print(f"Images                  : {len(files)}")
print(f"Attack success rate     : {100 * np.mean(succ):.2f}%")
print(f"Mean L2 (pixels in 0-1) : {np.mean(l2s):.4f}")
print(f"Mean Linf (0-255 scale) : {np.mean(linfs):.2f}")
print(f"Mean PSNR               : {np.mean(psnrs):.2f} dB")
if ssims: print(f"Mean SSIM               : {np.mean(ssims):.4f}")
with open(RESULTS_CSV) as f:
    q = [int(r["queries"]) for r in csv.DictReader(f)]
print(f"Mean queries per image  : {np.mean(q):.0f}")